# 🕵️ Dia 02 — Disciplina de evidência, da coleta ao registro de custódia

**Investigador Cyber 90 Dias** · Fase 1 · Rank Jutsu `D`

> Baseado no *Day 2* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-1-foundations/day-02.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

São 11h38. Você tirou um print do site falso `serraazul-pix.example` e colou no canal. Rafa: *"Agora prova que esse print é daquele servidor, daquele horário, e que ninguém mexeu nele até a delegacia receber."*

E tem um detalhe esquisito: a página diz só *"Sistema em manutenção"*.

Neste notebook você vai:

1. **Ver o efeito avalanche**: um caractere muda o SHA-256 inteiro.
2. **Reescrever o `sha256sum -c` em Python** e rodar nas pastas da coleta da Operação Isca (uma delas foi adulterada).
3. **Jogar o quiz "o hash prova isso?"** (8 afirmações).
4. **Gerar o registro de custódia** com hora UTC e hashes colados, nunca redigitados.

🐍 Só biblioteca padrão do Python. Nada para instalar. Rode as células de cima para baixo (Shift+Enter). Funciona sem internet: toda célula que tenta a rede tem plano B embutido.

## 1. O efeito avalanche

Um *hash* é a impressão digital dos bytes. Mude **um** caractere e o resultado muda por completo, não "um pouquinho". É isso que torna o hash útil como lacre: não existe adulteração "pequena demais" para ele perceber.

In [ ]:
import hashlib   # biblioteca padrão: SHA-256, SHA-1, MD5...

texto_original = "Sistema em manutenção"      # trecho da página falsa
texto_alterado = "Sistema em manutençã0"      # um único caractere trocado (o por zero)

# .encode() transforma texto em bytes: o hash trabalha sobre BYTES, não sobre "letras"
h1 = hashlib.sha256(texto_original.encode("utf-8")).hexdigest()
h2 = hashlib.sha256(texto_alterado.encode("utf-8")).hexdigest()

print("original:", h1)
print("alterado:", h2)

# Quantas das 64 posições hexadecimais continuaram iguais? (por acaso, ~1/16 delas)
iguais = sum(a == b for a, b in zip(h1, h2))
print(f"\nPosições iguais: {iguais}/64 — praticamente tudo mudou.")

## 2. Os arquivos da coleta

A Vigia coletou o site falso com `curl` a partir da VM de análise (saída por VPN de pesquisa). A pasta tem três partes, como no original do Dia 2:

- `original/` — os bytes como chegaram (somente leitura no disco real);
- `notas/` — `SHA256SUMS`, `transfer.txt` (com o IP remoto) e a hora de início;
- `trabalho/` — a cópia de trabalho. **Alguém mexeu nela.** Vamos descobrir onde.

A célula abaixo tenta baixar os arquivos do repositório; sem internet, usa a cópia embutida.

In [ ]:
# ------------------------------------------------------------
# DADOS DO CASO (Operação Isca — tudo fictício)
# Primeiro tentamos baixar do repositório no GitHub; se não houver
# internet (ou o repo estiver fora do ar), usamos a cópia embutida.
# ------------------------------------------------------------
import urllib.request   # baixa arquivos pela web, sem instalar nada

BASE = "https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/main/caso-continuo/dados/dia-02/"

# Cópias embutidas: o plano B. São idênticas aos arquivos do repo.
EMBUTIDO = {}
EMBUTIDO['original/headers.txt'] = 'HTTP/1.1 301 Moved Permanently\r\nDate: Mon, 05 Oct 2026 14:41:09 GMT\r\nServer: nginx\r\nLocation: https://serraazul-pix.example/\r\nContent-Length: 162\r\n\r\nHTTP/1.1 200 OK\r\nDate: Mon, 05 Oct 2026 14:41:10 GMT\r\nServer: nginx\r\nContent-Type: text/html; charset=utf-8\r\nContent-Length: 406\r\nSet-Cookie: isk_sess=7f3a91c2e0; Path=/; HttpOnly\r\n\r\n'
EMBUTIDO['original/page.html'] = '<!DOCTYPE html>\n<html lang="pt-BR">\n<head>\n<meta charset="utf-8">\n<title>Cooperativa Serra Azul</title>\n<meta name="viewport" content="width=device-width, initial-scale=1">\n<!-- IscaKit v2.3 - pescador -->\n</head>\n<body style="font-family:sans-serif;text-align:center;margin-top:80px">\n<h1>Cooperativa Serra Azul</h1>\n<p>Sistema em manuten&ccedil;&atilde;o. Tente novamente mais tarde.</p>\n</body>\n</html>\n'
EMBUTIDO['notas/transfer.txt'] = 'final_url=https://serraazul-pix.example/\nhttp_code=200\nremote_ip=203.0.113.77\nsize=406\n'
EMBUTIDO['notas/inicio-coleta.txt'] = '2026-10-05T14:41:08Z\n'
EMBUTIDO['notas/SHA256SUMS'] = '401f3e69b174c97671fdbcf14a85f26a1c706124e51081c69643f43cdcac8394  headers.txt\n04da3589a8d2767d03f975e322a207924f415e753b9039fac8f7b6b1973da578  page.html\n'
EMBUTIDO['trabalho/headers.txt'] = 'HTTP/1.1 301 Moved Permanently\r\nDate: Mon, 05 Oct 2026 14:41:09 GMT\r\nServer: nginx\r\nLocation: https://serraazul-pix.example/\r\nContent-Length: 162\r\n\r\nHTTP/1.1 200 OK\r\nDate: Mon, 05 Oct 2026 14:41:10 GMT\r\nServer: nginx\r\nContent-Type: text/html; charset=utf-8\r\nContent-Length: 406\r\nSet-Cookie: isk_sess=7f3a91c2e0; Path=/; HttpOnly\r\n\r\n'
EMBUTIDO['trabalho/page.html'] = '<!DOCTYPE html>\n<html lang="pt-BR">\n<head>\n<meta charset="utf-8">\n<title>Cooperativa Serra Azul</title>\n<meta name="viewport" content="width=device-width, initial-scale=1">\n<!-- IscaKit v2.3 - pescador -->\n</head>\n<body style="font-family:sans-serif;text-align:center;margin-top:80px">\n<h1>Cooperativa Serra Azul</h1>\n<p>Sistema em manuten&ccedil;&atilde;0. Tente novamente mais tarde.</p>\n</body>\n</html>\n'

def carregar(nome):
    """Devolve o texto do arquivo: da rede se der, senão da cópia embutida."""
    try:
        with urllib.request.urlopen(BASE + nome, timeout=5) as r:   # timeout: não travar a aula
            texto = r.read().decode("utf-8")
        if texto == EMBUTIDO[nome]:                                   # confere com a cópia local
            print(f"🌐 {nome}: baixado do GitHub (idêntico à cópia embutida)")
            return texto
        print(f"⚠️ {nome}: a versão do GitHub difere da embutida; usando a embutida (é a que os hashes do notebook esperam)")
        return EMBUTIDO[nome]
    except Exception as erro:                                         # sem rede, 404, proxy...
        print(f"📦 {nome}: usando a cópia embutida ({type(erro).__name__})")
        return EMBUTIDO[nome]

print(f"{len(EMBUTIDO)} arquivo(s) do caso disponíveis:", ", ".join(EMBUTIDO))

In [ ]:
from pathlib import Path   # caminhos de arquivo que funcionam em qualquer sistema

# Recria a pasta do caso aqui no Colab (ou no seu computador), arquivo por arquivo.
CASO = Path("casos/ISCA-D02")
for nome in EMBUTIDO:
    destino = CASO / nome
    destino.parent.mkdir(parents=True, exist_ok=True)          # cria original/, notas/, trabalho/
    destino.write_bytes(carregar(nome).encode("utf-8"))        # grava como bytes, sem mexer em nada

print()
print("Início da coleta (UTC):", (CASO / "notas/inicio-coleta.txt").read_text().strip())
print((CASO / "notas/transfer.txt").read_text())

### ⏰ A hora da coleta foi depois da autorização?

O contrato foi assinado às **11h00 BRT** (Dia 01). Coleta antes disso seria coleta sem autorização. Vamos converter UTC → BRT (UTC-3) com `datetime`, sem fazer conta de cabeça.

In [ ]:
from datetime import datetime, timezone, timedelta

BRT = timezone(timedelta(hours=-3), "BRT")   # Brasil não tem mais horário de verão: UTC-3 fixo

inicio_utc = datetime.strptime((CASO / "notas/inicio-coleta.txt").read_text().strip(),
                               "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=timezone.utc)   # o "Z" quer dizer UTC
contrato = datetime(2026, 10, 5, 11, 0, tzinfo=BRT)                                 # fato fixo do caso

print("Coleta em UTC:", inicio_utc.isoformat())
print("Coleta em BRT:", inicio_utc.astimezone(BRT).isoformat())
print("Depois do contrato?", "✅ sim" if inicio_utc > contrato else "❌ NÃO — pare e fale com o Rafa")

## 3. O `sha256sum -c`, em Python

O comando `sha256sum -c SHA256SUMS` lê cada linha (`hash  nome`), recalcula o hash do arquivo e diz `OK` ou `FAILED`. Vamos escrever o mesmo, linha a linha.

In [ ]:
def sha256_arquivo(caminho):
    """SHA-256 de um arquivo, lendo em blocos (funciona até com arquivos de vários GB)."""
    h = hashlib.sha256()
    with open(caminho, "rb") as f:                 # "rb" = bytes crus; nunca abra evidência como texto
        for bloco in iter(lambda: f.read(65536), b""):   # lê 64 KB por vez até o fim
            h.update(bloco)                        # vai alimentando o hash
    return h.hexdigest()


def conferir(pasta, arquivo_somas):
    """Equivalente a: cd pasta && sha256sum -c arquivo_somas. Devolve True se tudo OK."""
    tudo_ok = True
    for linha in Path(arquivo_somas).read_text().splitlines():
        if not linha.strip():
            continue                               # pula linhas vazias
        esperado, nome = linha.split(maxsplit=1)   # formato: "<hash><2 espaços><nome>"
        nome = nome.lstrip("*")                    # o "*" indica modo binário em alguns sistemas
        atual = sha256_arquivo(Path(pasta) / nome)
        ok = (atual == esperado)
        tudo_ok &= ok
        print(f"{nome}: {'OK' if ok else 'FAILED'}")
    return tudo_ok


print("== original/");  conferir(CASO / "original", CASO / "notas/SHA256SUMS")
print("== trabalho/");  conferir(CASO / "trabalho", CASO / "notas/SHA256SUMS")

### 🔎 Onde está a adulteração?

O hash diz **que** mudou; não diz **onde**. Para isso usamos `difflib`, que compara linha a linha (é o `diff` do Python).

In [ ]:
import difflib

a = (CASO / "original/page.html").read_text().splitlines()
b = (CASO / "trabalho/page.html").read_text().splitlines()

for linha in difflib.unified_diff(a, b, "original/page.html", "trabalho/page.html", lineterm=""):
    print(linha)

# Agora restauramos a cópia de trabalho a partir do original, como manda o Passo 7.
import shutil
shutil.copyfile(CASO / "original/page.html", CASO / "trabalho/page.html")
print("\n♻️ Restaurado. Nova conferência:")
conferir(CASO / "trabalho", CASO / "notas/SHA256SUMS")

### 👀 O que a página e os cabeçalhos contam

Leia o que o servidor devolveu. Repare em três coisas: o `301` antes do `200`, o cookie `isk_sess` e o comentário HTML com o nome do kit.

In [ ]:
print((CASO / "original/headers.txt").read_text().replace("\r", ""))

# Comentários HTML (<!-- ... -->) não aparecem na tela, mas estão nos bytes. Print nenhum mostra isto.
import re
for comentario in re.findall(r"<!--(.*?)-->", (CASO / "original/page.html").read_text(), re.S):
    print("Comentário no HTML:", comentario.strip())
print("\n➡️ 'pescador' é registrado como ARTEFATO DO KIT, não como pessoa (Dia 01).")

## 4. 🎮 Quiz: o hash prova isso?

Oito afirmações sobre a coleta do site falso. Responda **S** (o hash prova) ou **N** (não prova). `MODO_JOGO = False` mostra só as respostas.

In [ ]:
AFIRMACOES = [
    ("O page.html que tenho hoje é idêntico ao que gravei às 14:41:08Z.", True,
     "É exatamente o que o hash prova: integridade desde a coleta."),
    ("O servidor mostra esta mesma página 'em manutenção' para todas as vítimas.", False,
     "Kits servem páginas diferentes por IP, navegador e idioma. O hash vale só para o que foi enviado A VOCÊ."),
    ("O conteúdo da página é verdadeiro.", False,
     "Hash mede integridade, não veracidade. Uma mentira íntegra continua mentira."),
    ("Ninguém editou a página antes de ela chegar até mim.", False,
     "O hash protege da coleta em diante. O que aconteceu antes está fora do alcance dele."),
    ("A cópia em trabalho/ foi alterada depois da coleta.", True,
     "O sha256sum -c deu FAILED: houve alteração depois que o hash foi calculado."),
    ("O 'pescador' do comentário HTML é o autor do golpe.", False,
     "Hash nenhum prova autoria. É um artefato do kit; atribuição é trabalho da polícia."),
    ("Se eu redigitar o hash à mão no relatório, ele continua valendo como prova.", False,
     "Um caractere errado e a conferência falha (ou pior, ninguém confere). Cole, não digite."),
    ("O arquivo headers.txt não mudou desde a coleta.", True,
     "Deu OK na conferência, nas duas pastas."),
]

MODO_JOGO = True   # False = só mostra o gabarito, sem perguntar

acertos = 0
for i, (frase, prova, porque) in enumerate(AFIRMACOES, start=1):
    print(f"\n[{i}/{len(AFIRMACOES)}] {frase}")
    if MODO_JOGO:
        resp = ""
        while resp not in ("S", "N"):                       # insiste até vir S ou N
            resp = input("   O hash prova isso? (S/N): ").strip().upper()
        acertou = (resp == "S") == prova
        acertos += acertou
        print("   ✅ Acertou!" if acertou else "   ❌ Não foi dessa vez.")
    print(f"   Resposta: {'prova' if prova else 'NÃO prova'} — {porque}")

if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(AFIRMACOES)}", "— Checkpoint BR cumprido. 🥷" if acertos >= 6 else "— meta: 6 de 8.")

## 5. 📝 Registro de custódia

Preencha o formulário e rode. Os hashes e o IP remoto são **lidos dos arquivos**, nunca digitados: é a regra do original ("não redigite hashes à mão").

In [ ]:
#@title Registro de custódia { display-mode: "form" }
caso_id = "VIGIA-2026-001 / Operação Isca"  #@param {type:"string"}
coletado_por = "Analista júnior, Vigia (VM vigia-lab01)"  #@param {type:"string"}
metodo = "curl 8.5.0 -sS -L -D -o -w, via VPN de pesquisa"  #@param {type:"string"}
terceiro = "Wayback Machine (Save Page Now) + ata notarial solicitada pela cooperativa"  #@param {type:"string"}

from datetime import datetime, timezone

agora = lambda: datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")   # hora UTC com Z
coleta = (CASO / "notas/inicio-coleta.txt").read_text().strip()
transfer = dict(l.split("=", 1) for l in (CASO / "notas/transfer.txt").read_text().split())  # chave=valor
h_head = sha256_arquivo(CASO / "original/headers.txt")
h_page = sha256_arquivo(CASO / "original/page.html")

registro = f"""# Registro de custódia de evidência: {caso_id}

| Item | Descrição | Fonte | Coletado (UTC) | Coletado por | Método | SHA-256 | Guardado em |
|---|---|---|---|---|---|---|---|
| E1 | Cabeçalhos HTTP | {transfer['final_url']} (IP remoto {transfer['remote_ip']}) | {coleta} | {coletado_por} | {metodo} | {h_head} | original/headers.txt |
| E2 | HTML da página | idem | {coleta} | {coletado_por} | idem | {h_page} | original/page.html |
| E3 | Captura de terceiro | {terceiro} | (preencher) | terceiro | Save Page Now / ata | n/a (URL ou nº da ata) | notas/terceiro.txt |

## Eventos de manuseio
| Hora (UTC) | Item | Ação | Por | Hash verificado? |
|---|---|---|---|---|
| {coleta} | E1, E2 | Coletado, hash calculado, definido como somente leitura | {coletado_por} | sim |
| {agora()} | E2 | Conferência de trabalho/: page.html FAILED (1 caractere alterado) | {coletado_por} | não (FAILED) |
| {agora()} | E2 | Cópia de trabalho restaurada a partir de original/ | {coletado_por} | sim (OK) |

> O hash prova que estes arquivos não mudaram desde {coleta}. Não prova que o servidor mostra a mesma página às vítimas.
"""

Path("notas").mkdir(exist_ok=True)
saida = Path("notas/registro-de-custodia-isca.md")
saida.write_text(registro, encoding="utf-8")
print(registro)
print("💾 Salvo em", saida)

try:
    from google.colab import files      # só existe dentro do Colab
    files.download(str(saida))
except ImportError:
    print("(Fora do Colab: o arquivo ficou na pasta notas/)")

## ✅ Antes de fechar o Dia 2

- [ ] Viu o efeito avalanche e sabe explicar por que ele importa
- [ ] Achou o arquivo adulterado em `trabalho/` e o caractere trocado
- [ ] Quiz com pelo menos 6/8
- [ ] `notas/registro-de-custodia-isca.md` gerado
- [ ] Fez também a prática original (`~/cases/P1-D02` com a página da IANA)

**Amanhã (Dia 3):** a página dizia "manutenção" para você. O que aconteceu na rede quando você pediu? Rafa vai abrir o Wireshark.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>